# Banana / Plantain Leaf Disease Classifier — training + ONNX export

Trains an image classifier on the **BananaLSD** (Banana Leaf Spot Diseases)
dataset — 937 field-collected images across 4 classes: Sigatoka, Cordana,
Pestalotiopsis, and Healthy — and exports:

- `banana_classifier.pt` — PyTorch checkpoint (state_dict + class map + config)
- `banana_classifier.onnx` — ONNX export for CPU/edge inference
- `banana_classifier.int8.onnx` — dynamically quantized INT8 ONNX (optional)

**Note on "plantain":** there's no dataset that specifically labels itself
"plantain" — plantain and banana are both *Musa* species and every ag dataset
in this space just says "banana". BananaLSD's Sigatoka/Cordana/Pestalotiopsis
labels transfer reasonably well to plantain leaves since they're the same
pathogens, but if plantain-specific accuracy matters for the drone pipeline,
plan to fine-tune further on your own labeled plantain frames later — 937
images is a small dataset and this model will not generalize as confidently
as the cassava one.

**Run this on Colab with a GPU runtime** (Runtime → Change runtime type → GPU).


In [ ]:
!pip install -q timm onnx onnxruntime kaggle


## 1. Get the dataset

Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when prompted.


In [ ]:
from google.colab import files
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # select kaggle.json
for fname in uploaded:
    shutil.move(fname, f"/root/.kaggle/{fname}")
os.chmod("/root/.kaggle/kaggle.json", 0o600)


In [ ]:
DATA_DIR = "/content/banana-data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d shifatearman/bananalsd -p {DATA_DIR}
!cd {DATA_DIR} && unzip -q -o bananalsd.zip -d .
!find {DATA_DIR} -maxdepth 3 -type d


## 2. Imports & config

In [ ]:
import time
import copy
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# BananaLSD ships as class-named subfolders (e.g. .../Original Images/<class>/*.jpg).
# Point ROOT_DIR at whichever folder actually contains one subfolder per class —
# print the tree above and adjust this path if Kaggle's layout differs.
import glob

candidates = [p for p in glob.glob(f"{DATA_DIR}/**/", recursive=True)
              if len(glob.glob(p + "*")) > 1 and
              all(Path(sub).is_dir() for sub in glob.glob(p + "*"))]
print("candidate class-folder roots:")
for c in candidates:
    print(" ", c, "->", [Path(s).name for s in glob.glob(c + "*")])


In [ ]:
# Set this explicitly once you've confirmed the right folder from the cell above.
ROOT_DIR = candidates[0] if candidates else DATA_DIR
print("Using ROOT_DIR =", ROOT_DIR)


In [ ]:
class CFG:
    img_size = 380
    batch_size = 16          # small dataset — keep batches modest
    num_workers = 2
    epochs = 25               # more epochs since the dataset is small; watch for overfitting
    lr = 3e-4
    weight_decay = 5e-4       # stronger reg — small dataset
    label_smoothing = 0.1
    val_frac = 0.2
    model_name = "efficientnet_b0"
    class_names = None        # filled in after ImageFolder discovers classes

cfg = CFG()


## 3. Dataset + transforms

Because BananaLSD is small (937 raw images), we lean harder on augmentation
than the cassava notebook, and skip the pre-augmented copies Kaggle ships
(they're derived from the same 937 source images, so mixing them into
train/val risks leaking near-duplicates across the split).


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=0, shear=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load once with val_tfms just to discover classes + indices; we'll wrap
# train indices with a version using train_tfms below.
base_ds = datasets.ImageFolder(ROOT_DIR, transform=val_tfms)
cfg.class_names = {v: k for k, v in base_ds.class_to_idx.items()}
print(cfg.num_classes if hasattr(cfg, "num_classes") else "classes:", cfg.class_names)

targets = np.array(base_ds.targets)
idx_all = np.arange(len(base_ds))
train_idx, val_idx = train_test_split(
    idx_all, test_size=cfg.val_frac, stratify=targets, random_state=SEED
)

train_ds_full = datasets.ImageFolder(ROOT_DIR, transform=train_tfms)
train_ds = Subset(train_ds_full, train_idx)
val_ds = Subset(base_ds, val_idx)

cfg.num_classes = len(cfg.class_names)
print(f"train: {len(train_ds)}  val: {len(val_ds)}  classes: {cfg.num_classes}")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                           num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)


In [ ]:
fig, axes = plt.subplots(1, cfg.num_classes, figsize=(4*cfg.num_classes, 4))
seen = set()
for img_path, label in base_ds.samples:
    if label in seen:
        continue
    seen.add(label)
    img = Image.open(img_path)
    ax = axes[label] if cfg.num_classes > 1 else axes
    ax.imshow(img)
    ax.set_title(cfg.class_names[label], fontsize=10)
    ax.axis("off")
    if len(seen) == cfg.num_classes:
        break
plt.tight_layout(); plt.show()


## 4. Model

In [ ]:
def build_model():
    model = timm.create_model(cfg.model_name, pretrained=True, num_classes=cfg.num_classes)
    return model.to(device)

model = build_model()
sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Training loop

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(imgs)
                loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += imgs.size(0)

    return total_loss / n, correct / n


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0.0
best_state = None
patience, no_improve = 6, 0   # early stopping — small dataset overfits fast

for epoch in range(cfg.epochs):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    print(f"epoch {epoch+1:02d}/{cfg.epochs}  "
          f"train_loss {tr_loss:.4f} train_acc {tr_acc:.4f}  "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}  "
          f"({time.time()-t0:.0f}s)")

    if no_improve >= patience:
        print(f"early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
        break

print(f"best val acc: {best_acc:.4f}")
model.load_state_dict(best_state)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()


## 6. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds,
                             target_names=[cfg.class_names[i] for i in range(cfg.num_classes)]))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(cfg.num_classes)); ax.set_yticks(range(cfg.num_classes))
ax.set_xticklabels(range(cfg.num_classes)); ax.set_yticklabels(range(cfg.num_classes))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(cfg.num_classes):
    for j in range(cfg.num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im)
plt.tight_layout(); plt.show()


## 7. Save the PyTorch checkpoint (`.pt`)

In [ ]:
OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_name": cfg.model_name,
    "img_size": cfg.img_size,
    "num_classes": cfg.num_classes,
    "class_names": cfg.class_names,
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "best_val_acc": best_acc,
}

pt_path = OUT_DIR / "banana_classifier.pt"
torch.save(checkpoint, pt_path)
print("saved", pt_path)


## 8. Export to ONNX

In [ ]:
onnx_path = OUT_DIR / "banana_classifier.onnx"

model.eval()
dummy_input = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=device)

torch.onnx.export(
    model,
    dummy_input,
    str(onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
)
print("saved", onnx_path)


In [ ]:
import onnx
import onnxruntime as ort

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

with torch.no_grad():
    torch_out = model(dummy_input).cpu().numpy()

ort_out = ort_session.run(None, {"input": dummy_input.cpu().numpy()})[0]

max_diff = np.abs(torch_out - ort_out).max()
print(f"max abs diff between PyTorch and ONNXRuntime outputs: {max_diff:.6f}")
assert max_diff < 1e-3, "ONNX export mismatch — investigate before trusting this model"
print("ONNX export verified ✓")


## 9. (Optional) INT8 dynamic quantization

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

int8_path = OUT_DIR / "banana_classifier.int8.onnx"
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(int8_path),
    weight_type=QuantType.QInt8,
)
print("saved", int8_path)
print("fp32 size:", onnx_path.stat().st_size / 1e6, "MB")
print("int8 size:", int8_path.stat().st_size / 1e6, "MB")


In [ ]:
int8_session = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])

correct, n = 0, 0
for imgs, labels in val_loader:
    out = int8_session.run(None, {"input": imgs.numpy()})[0]
    preds = out.argmax(1)
    correct += (preds == labels.numpy()).sum()
    n += len(labels)

print(f"INT8 ONNX val accuracy: {correct/n:.4f}  (fp32 was {best_acc:.4f})")


## 10. Download the artifacts

In [ ]:
from google.colab import files as colab_files

for f in [pt_path, onnx_path, int8_path]:
    colab_files.download(str(f))
